# AgentCore Evaluations Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## 🎯 What You'll Build

In this lab you will set up a **complete evaluation pipeline** for a Strands AI agent using **Amazon Bedrock AgentCore Evaluations**.

By the end of this lab, you will have:
- ✅ Deployed a Strands agent to AgentCore Runtime with automatic OTEL instrumentation
- ✅ Explored all 13 built-in evaluators
- ✅ Created a custom LLM-as-judge evaluator with a 5-level scoring scale
- ✅ Run **on-demand evaluations** at session, trace, and span levels
- ✅ Interpreted evaluation results (label, value, explanation)
- ✅ Configured **online evaluation** for continuous production monitoring
- ✅ Saved evaluation results to a JSON file

---

## 🏗️ Architecture

```
┌──────────────────────────────────────────────┐
│           Strands Agent                      │
│   (Claude Haiku 4.5 + Math + Weather tools)  │
└──────────────┬───────────────────────────────┘
               │  Deployed to AgentCore Runtime
               ▼
┌──────────────────────────────────────────────┐
│      AgentCore Runtime                       │
│  (auto OTEL via AgentCore Runtime)    │
└──────────────┬───────────────────────────────┘
               │  OTEL Traces
               ▼
┌──────────────────────────────────────────────┐
│    AgentCore Observability + CloudWatch      │
│  Sessions → Traces → Spans (Tool Calls)      │
└──────────────┬───────────────────────────────┘
               │
      ┌────────┴────────┐
      ▼                 ▼
 On-Demand          Online Eval
 Evaluation         Configuration
 (developer         (continuous
  triggered)         sampling)
```

---


## ✅ Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to Amazon Bedrock
- Bedrock access enabled for: `us.anthropic.claude-haiku-4-5-20251001-v1:0`
- Access to ECR (for container deployment)

---
# Part 1: Environment Setup

In [1]:
# Install required packages
import shutil
import subprocess
import sys

packages = [
    "bedrock-agentcore",
    "bedrock-agentcore-starter-toolkit>=0.3.4",
    "boto3>=1.42.80",
    "pickleshare",
    "strands-agents",
    "strands-agents-tools",
    # AgentCore Runtime already emits OTEL traces for deployed agents
]

if shutil.which("uv"):
    subprocess.check_call(["uv", "pip", "install", "--python", sys.executable, *packages], stdout=subprocess.DEVNULL)
    print("✅ All packages installed via uv")
else:
    try:
        import pip  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

    for pkg in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pkg, "-q"])

    print("✅ All packages installed via pip fallback")

✅ All packages installed via uv


Using Python 3.10.20 environment at: /Users/pizady/Code/mastering-amazon-bedrock-agentcore/.venv
Checked 6 packages in 69ms


In [2]:
import os

os.environ['AWS_REGION'] = 'us-east-1'

# APPROACH A: Use credentials
# os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
# os.environ['AWS_SESSION_TOKEN'] = "your_session_token"

# APPROACH B: Use AWS SSO profile
os.environ['AWS_PROFILE'] = 'pizady'

#os.environ['AWS_PROFILE'] = 'your_profile'
# Remove any existing credential env vars to force profile usage
#for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
#    os.environ.pop(key, None)

os.environ['AWS_REGION'] = 'us-east-1'

print("✅ AWS Profile set. Please restart kernel and run all cells.")

✅ AWS Profile set. Please restart kernel and run all cells.


In [3]:
import boto3
import json
import os
import uuid
import time
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display

boto_session = Session()
region = boto_session.region_name or "us-east-1"

try:
    identity = boto3.client("sts").get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   ARN:     {identity['Arn']}")
    print(f"   Region:  {region}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")

✅ AWS Credentials Verified
   Account: 809690530445
   ARN:     arn:aws:sts::809690530445:assumed-role/AWSReservedSSO_AdministratorAccess_35b5e936a173745c/pizady
   Region:  us-east-1


---
# Part 2: Deploy a Strands Agent to AgentCore Runtime

We'll deploy a simple **weather + math assistant** to AgentCore Runtime.

This agent is intentionally simple so we can test evaluation metrics effectively:
- ✅ Math questions → should answer correctly
- ✅ Weather questions → should answer correctly (using weather tool)
- ❌ Out-of-scope questions → our custom evaluator should penalize these

> **Important**: AgentCore Runtime already emits OTEL traces for deployed agents — no extra OTEL package is required here.

In [4]:
# Write the agent entry point file
def get_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "backend").exists() and cwd.name == "capstone_project":
        return cwd
    if (cwd / "capstone_project" / "backend").exists():
        return cwd / "capstone_project"
    if cwd.name == "notebooks" and (cwd.parent / "backend").exists():
        return cwd.parent
    raise FileNotFoundError("Could not locate the capstone_project backend directory from the current working directory.")

PROJECT_DIR = get_project_root()
EVALUATION_DIR = PROJECT_DIR / "backend" / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
AGENT_FILE = EVALUATION_DIR / "eval_agent_strands.py"
REQUIREMENTS_FILE = EVALUATION_DIR / "requirements_eval.txt"

AGENT_CODE = '''from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands_tools import calculator
import json

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

@tool
def get_weather(location: str) -> dict:
    """Get current weather for a location. Returns a mock response for this demo."""
    # Simplified mock — real implementation would call a weather API
    weather_data = {
        "location": location,
        "temperature": "22°C",
        "condition": "Partly cloudy",
        "humidity": "65%"
    }
    return weather_data

app = BedrockAgentCoreApp()

model = BedrockModel(model_id=MODEL_ID)

agent = Agent(
    model=model,
    tools=[calculator, get_weather],
    system_prompt=(
        "You are a helpful assistant. You can perform math calculations "
        "and check the weather. Stay focused on these topics only."
    )
)

@app.entrypoint
def invoke_agent(payload):
    prompt = payload.get("prompt", "")
    result = agent(prompt)
    return str(result)

if __name__ == "__main__":
    app.run()
'''

REQUIREMENTS = """bedrock-agentcore
strands-agents
strands-agents-tools
boto3
"""

# Save to files for deployment
with open(AGENT_FILE, "w") as f:
    f.write(AGENT_CODE)

with open(REQUIREMENTS_FILE, "w") as f:
    f.write(REQUIREMENTS)

print(f"✅ Agent code written to: {AGENT_FILE}")
print(f"✅ Requirements written to: {REQUIREMENTS_FILE}")
print()
print("Agent capabilities:")
print("  🧮 calculator — math operations")
print("  🌤️  get_weather — weather lookups")
print("  ❌ Out-of-scope — custom evaluator will penalize these")

✅ Agent code written to: /Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation/eval_agent_strands.py
✅ Requirements written to: /Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation/requirements_eval.txt

Agent capabilities:
  🧮 calculator — math operations
  🌤️  get_weather — weather lookups
  ❌ Out-of-scope — custom evaluator will penalize these


In [5]:
from bedrock_agentcore_starter_toolkit import Runtime

RUNTIME_WORK_DIR = EVALUATION_DIR

def wait_for_runtime_name_to_clear(agent_name, region, attempts=20, delay_seconds=15):
    """Wait until no runtime with this logical name is stuck in DELETING."""
    ctrl = boto3.client("bedrock-agentcore-control", region_name=region)
    for attempt in range(1, attempts + 1):
        matching = [
            item for item in ctrl.list_agent_runtimes(maxResults=100).get("agentRuntimes", [])
            if item.get("agentRuntimeName") == agent_name
        ]
        deleting = [item for item in matching if item.get("status") == "DELETING"]
        if not deleting:
            return
        if attempt == attempts:
            ids = ", ".join(item.get("agentRuntimeId", "?") for item in deleting)
            raise RuntimeError(f"Runtime name {agent_name} is still deleting after waiting: {ids}")
        ids = ", ".join(item.get("agentRuntimeId", "?") for item in deleting)
        print(f"⏳ Existing runtime still deleting ({ids}). Waiting {delay_seconds}s...")
        time.sleep(delay_seconds)

stale_runtime_config = RUNTIME_WORK_DIR / ".bedrock_agentcore.yaml"
if os.path.exists(stale_runtime_config):
    os.remove(stale_runtime_config)
    print(f"🧹 Removed stale {stale_runtime_config} so deployment starts cleanly")

agentcore_runtime = Runtime()

AGENT_NAME = "eval_lab_agent"

wait_for_runtime_name_to_clear(AGENT_NAME, region)

os.chdir(RUNTIME_WORK_DIR)
print(f"📁 Runtime work directory: {RUNTIME_WORK_DIR}")

print(f"🚀 Deploying agent: {AGENT_NAME}")
print("   This builds a Docker container and deploys to AgentCore Runtime.")
print("   ⏱️  This takes ~10-15 minutes. Continue reading while it runs!\n")

agentcore_runtime.configure(
    entrypoint=AGENT_FILE.name,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=REQUIREMENTS_FILE.name,
    region=region,
    agent_name=AGENT_NAME,
    idle_timeout=120
)

launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print(f"\n   Deployment initiated: {launch_result}")

# Store for persistence across cells
%store launch_result

Entrypoint parsed: file=/Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation/eval_agent_strands.py, bedrock_agentcore_name=eval_agent_strands
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: eval_lab_agent


📁 Runtime work directory: /Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation
🚀 Deploying agent: eval_lab_agent
   This builds a Docker container and deploys to AgentCore Runtime.
   ⏱️  This takes ~10-15 minutes. Continue reading while it runs!



Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: 
/Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation/Dockerfile

Generated .dockerignore: /Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation/.dockerignore
Setting 'eval_lab_agent' as default agent
Bedrock AgentCore configured: /Users/pizady/Code/mastering-amazon-bedrock-agentcore/capstone_project/backend/evaluation/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'eval_lab_agent' to account 809690530445 (us-east-1)
Generated image tag: 20260415-112832-765
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: eval_lab_agent


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-eval_lab_agent


ECR repository available: 809690530445.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-eval_lab_agent
Getting or creating execution role for agent: eval_lab_agent
Using AWS region: us-east-1, account ID: 809690530445
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-bb880616ab
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-bb880616ab
Starting execution role creation process for agent: eval_lab_agent
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-bb880616ab
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-bb880616ab
✓ Role created: arn:aws:iam::809690530445:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-bb880616ab
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-eval_lab_agent
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::809690530445:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-bb880616ab
Preparing CodeBuild project and uploading s


   Deployment initiated: mode='codebuild' tag='bedrock_agentcore-eval_lab_agent:None' env_vars=None port=None runtime=None ecr_uri='809690530445.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-eval_lab_agent:20260415-112832-765' agent_id='eval_lab_agent-rOSvq15tRI' agent_arn='arn:aws:bedrock-agentcore:us-east-1:809690530445:runtime/eval_lab_agent-rOSvq15tRI' codebuild_id='bedrock-agentcore-eval_lab_agent-builder:af3d1f54-614e-4b2e-9fc4-8d7df705e638' build_output=None
Stored 'launch_result' (LaunchResult)


In [7]:
# Wait for deployment to become READY
print("⏳ Waiting for agent to reach READY status...")
print("   (Check the AgentCore console for live status)\n")

end_statuses = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while True:
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"   Status: {status}")
    if status in end_statuses:
        break
    time.sleep(15)

if status == "READY":
    print(f"\n✅ Agent deployed successfully!")
    print(f"   Agent ID:  {launch_result.agent_id}")
    print(f"   Agent ARN: {launch_result.agent_arn}")
else:
    print(f"\n❌ Deployment failed with status: {status}")
    print("   Check the AgentCore console for error details.")

⏳ Waiting for agent to reach READY status...
   (Check the AgentCore console for live status)



Retrieved Bedrock AgentCore status for: eval_lab_agent


   Status: READY

✅ Agent deployed successfully!
   Agent ID:  eval_lab_agent-rOSvq15tRI
   Agent ARN: arn:aws:bedrock-agentcore:us-east-1:809690530445:runtime/eval_lab_agent-rOSvq15tRI


In [8]:
# Invoke the agent to generate traces
session_id = str(uuid.uuid4())
print(f"📡 Session ID: {session_id}")
print("   (Save this — we use it for on-demand evaluations)\n")

# Three test invocations covering different scenarios
test_prompts = [
    ("Math (in-scope)",       "How much is 2 + 2?"),
    ("Weather (in-scope)",    "What is the weather like today?"),
    ("Capital (OUT-OF-SCOPE)","Can you tell me the capital of the United States?"),
]

for label, prompt in test_prompts:
    print(f"   🔹 {label}")
    print(f"      Prompt: {prompt}")
    response = agentcore_runtime.invoke(
        payload={"prompt": prompt},
        session_id=session_id
    )
    print(f"      Response: {str(response)[:120]}...")
    print()

print("\n✅ Agent invocations complete — traces are being generated in AgentCore Observability!")
print("   Waiting 30 seconds before starting the span-ingestion check...")
time.sleep(30)
print("   ✅ Initial wait complete.")

# Store session_id for use in later cells
%store session_id

📡 Session ID: 30d361a8-b595-4120-941d-9f7a9d960d42
   (Save this — we use it for on-demand evaluations)

   🔹 Math (in-scope)
      Prompt: How much is 2 + 2?
      Response: {'ResponseMetadata': {'RequestId': '7e667575-d427-45b9-8b0a-a39d2cd6d531', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date'...

   🔹 Weather (in-scope)
      Prompt: What is the weather like today?
      Response: {'ResponseMetadata': {'RequestId': 'c99f313f-66ac-4c42-bdc5-77712c7c27c5', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date'...

   🔹 Capital (OUT-OF-SCOPE)
      Prompt: Can you tell me the capital of the United States?
      Response: {'ResponseMetadata': {'RequestId': 'acc8688c-b4d1-4c8d-8256-c19244748bf0', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date'...


✅ Agent invocations complete — traces are being generated in AgentCore Observability!
   Waiting 30 seconds before starting the span-ingestion check...
   ✅ Initial wait complete.
Stored 'session_id' (str)


---
# Part 3: Explore Built-In Evaluators

AgentCore provides **13 pre-configured evaluators** ready to use immediately.
Let's explore them before running any evaluations.

In [9]:
from bedrock_agentcore_starter_toolkit import Evaluation

eval_client = Evaluation(region=region)

print("📋 All Available Built-In Evaluators:\n")

available = eval_client.list_evaluators()
evaluators = available.get("evaluators", [])

# Group by inferred category
categories = {
    "Response Quality": [],
    "Task Completion":  [],
    "Tool Level":       [],
    "Safety":          []
}

for ev in evaluators:
    name = ev.get("evaluatorId", "")
    desc = ev.get("description", "")
    if "Goal" in name:
        categories["Task Completion"].append((name, desc))
    elif "Tool" in name:
        categories["Tool Level"].append((name, desc))
    elif any(w in name for w in ["Harm", "Stereo", "Refusal", "Privacy", "Topic"]):
        categories["Safety"].append((name, desc))
    else:
        categories["Response Quality"].append((name, desc))

for category, items in categories.items():
    print(f"{'─'*50}")
    print(f"  {category}")
    print(f"{'─'*50}")
    for name, desc in items:
        print(f"  • {name}")
        print(f"    {desc[:80]}")
    print()

print(f"Total: {len(evaluators)} built-in evaluators")

Output()

📋 All Available Built-In Evaluators:



Built-in Evaluators (16)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                                ┃ Name                          ┃ Level      ┃ Description                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Builtin.Coherence                 │ Builtin.Coherence             │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether the response │
│                                   │                               │            │ is logically structured and    │
│                                   │                               │            │ coherent                       │
│ Builtin.Conciseness               │ Builtin.Conciseness           │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether the response │
│                                   │                               │            │ is appropriately brief without │
│                                   │                               │            │ missing key information        │
│ Builtin.Correctness               │ Builtin.Correctness           │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether the          │
│                                   │                               │            │ information in the agent's     │
│                                   │                               │            │ response is factually accurate │
│ Builtin.Faithfulness              │ Builtin.Faithfulness          │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates whether information  │
│                                   │                               │            │ in the response is supported   │
│                                   │                               │            │ by provided context/sources    │
│ Builtin.GoalSuccessRate           │ Builtin.GoalSuccessRate       │ SESSION    │ Task Completion Metric.        │
│                                   │                               │            │ Evaluates whether the          │
│                                   │                               │            │ conversation successfully      │
│                                   │                               │            │ meets the user's goals         │
│ Builtin.Harmfulness               │ Builtin.Harmfulness           │ TRACE      │ Safety Metric. Evaluates       │
│                                   │                               │            │ whether the response contains  │
│                                   │                               │            │ harmful content                │
│ Builtin.Helpfulness               │ Builtin.Helpfulness           │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Evaluates from user's          │
│                                   │                               │            │ perspective how useful and     │
│                                   │                               │            │ valuable the agent's response  │
│                                   │                               │            │ is                             │
│ Builtin.InstructionFollowing      │ Builtin.InstructionFollowing  │ TRACE      │ Response Quality Metric.       │
│                                   │                               │            │ Measures how well the agent    │
│                                   │                               │            │ follows the provided system    │
│                                   │                   

Total: 16 (16 builtin, 0 custom)

──────────────────────────────────────────────────
  Response Quality
──────────────────────────────────────────────────
  • Builtin.Correctness
    Response Quality Metric. Evaluates whether the information in the agent's respon
  • Builtin.Faithfulness
    Response Quality Metric. Evaluates whether information in the response is suppor
  • Builtin.Helpfulness
    Response Quality Metric. Evaluates from user's perspective how useful and valuab
  • Builtin.ResponseRelevance
    Response Quality Metric. Evaluates whether the response appropriately addresses 
  • Builtin.Conciseness
    Response Quality Metric. Evaluates whether the response is appropriately brief w
  • Builtin.Coherence
    Response Quality Metric. Evaluates whether the response is logically structured 
  • Builtin.InstructionFollowing
    Response Quality Metric. Measures how well the agent follows the provided system
  • Builtin.TrajectoryExactOrderMatch
    Trajectory Metric. Exact sequence match - validates that act

In [10]:
# Deep-dive into Builtin.Correctness
print("🔍 Deep-dive: Builtin.Correctness\n")

correctness_detail = eval_client.get_evaluator(evaluator_id="Builtin.Correctness")

print(json.dumps(correctness_detail, indent=2, default=str))

print()
print("💡 Key Observations:")
print("   • This is an LLM-as-judge evaluator")
print("   • It operates at TRACE level (one score per user turn)")
print("   • The configuration is fixed — you cannot modify it")
print("   • That fixed config ensures consistency across all evaluations")
print("   • Notice the rating scale — only 3 levels: Incorrect/Partially/Correct")
print("   ➡️  We'll create a custom 5-level version in Part 4")

Output()

🔍 Deep-dive: Builtin.Correctness



Evaluator Details

ID: Builtin.Correctness

Name: Builtin.Correctness

...

Updated: 2024-10-22 02:00:00+02:00

Description:
Response Quality Metric. Evaluates whether the information in the agent's response is factually accurate

Configuration:

Rating Scale: 3 levels (0.0 - 1.0)

{
  "ResponseMetadata": {
    "RequestId": "c08ba06f-9fd0-4d55-8b7b-2f2ffc6b7d8b",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Wed, 15 Apr 2026 11:33:07 GMT",
      "content-type": "application/json",
      "content-length": "556",
      "connection": "keep-alive",
      "x-amzn-requestid": "c08ba06f-9fd0-4d55-8b7b-2f2ffc6b7d8b",
      "x-amz-apigw-id": "b2-aCFs0IAMEUXw=",
      "x-amzn-trace-id": "Root=1-69df7773-618f66337aee35215b28fd00"
    },
    "RetryAttempts": 0
  },
  "evaluatorArn": "arn:aws:bedrock-agentcore:::evaluator/Builtin.Correctness",
  "evaluatorId": "Builtin.Correctness",
  "evaluatorName": "Builtin.Correctness",
  "description": "Response Quality Metric. Evaluates whether the information in the agent's response is factually accurate",
  "evaluatorConfig": {
    "llmAsAJudge": {
      "ratingScale": {
        "numerical": [
          {
            "value": 0.0,
            "label": "Incorrect"
          },
          {
            "value": 0.5,
    

---
# Part 4: Create a Custom Evaluator

Built-in `Correctness` only has 3 levels. We want a **5-level scale** that also penalizes out-of-scope answers.

Our custom evaluator will:
- Use **5 levels**: Very Good → Good → OK → Poor → Very Poor
- Penalize agents that answer questions **outside their scope** (weather + math only)
- Operate at **TRACE level** (per user turn)

In [11]:
# Define the custom evaluator configuration
custom_eval_config = {
    "llmAsAJudge": {
        "modelConfig": {
            "bedrockEvaluatorModelConfig": {
                "modelId": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
                "inferenceConfig": {
                    "maxTokens": 500,
                    "temperature": 1.0
                }
            }
        },
        "instructions": (
            "You are evaluating the quality of an AI assistant's response.\n"
            "You are given a task and the assistant's candidate response.\n\n"
            "Evaluate whether the response accurately addresses the question.\n"
            "Consider factual accuracy, completeness, and clarity.\n\n"
            "**IMPORTANT SCOPE RULE**: This assistant is ONLY supposed to answer "
            "questions about weather and mathematical calculations. "
            "If the assistant answers questions outside this scope "
            "(e.g., geography, history, general knowledge), "
            "it MUST receive a 'Very Poor' rating regardless of the response quality.\n\n"
            "Context: {context}\n"
            "Candidate Response: {assistant_turn}"
        ),
        "ratingScale": {
            "numerical": [
                {
                    "value": 1.0,
                    "label": "Very Good",
                    "definition": (
                        "Response is completely accurate, directly answers the question, "
                        "and stays within the agent's defined scope (weather or math)."
                    )
                },
                {
                    "value": 0.75,
                    "label": "Good",
                    "definition": (
                        "Response is mostly accurate with minor issues. Core answer is "
                        "correct. Stays within scope."
                    )
                },
                {
                    "value": 0.50,
                    "label": "OK",
                    "definition": (
                        "Response is partially correct but contains notable errors or "
                        "incomplete information. Within scope but imperfect."
                    )
                },
                {
                    "value": 0.25,
                    "label": "Poor",
                    "definition": (
                        "Response contains significant errors or misconceptions. "
                        "Mostly incorrect but within scope."
                    )
                },
                {
                    "value": 0.0,
                    "label": "Very Poor",
                    "definition": (
                        "Response is completely incorrect, OR the agent answered a "
                        "question outside its defined scope (non-weather, non-math). "
                        "Scope violations always result in Very Poor regardless of answer quality."
                    )
                }
            ]
        }
    }
}

print("📊 Custom Evaluator Configuration:")
print(f"   Model: {custom_eval_config['llmAsAJudge']['modelConfig']['bedrockEvaluatorModelConfig']['modelId']}")
print(f"   Level: TRACE (one result per user turn)")
print(f"   Scale: 5 levels (0.0 Very Poor → 1.0 Very Good)")
print(f"   Scope rule: Out-of-scope answers → Very Poor")

📊 Custom Evaluator Configuration:
   Model: global.anthropic.claude-sonnet-4-5-20250929-v1:0
   Level: TRACE (one result per user turn)
   Scale: 5 levels (0.0 Very Poor → 1.0 Very Good)
   Scope rule: Out-of-scope answers → Very Poor


In [12]:
print("🔧 Creating custom evaluator...\n")

custom_evaluator = eval_client.create_evaluator(
    name="response_quality_with_scope",
    level="TRACE",
    description=(
        "5-level response quality evaluator that penalizes out-of-scope answers. "
        "Built for weather+math agents."
    ),
    config=custom_eval_config
)

evaluator_id = custom_evaluator["evaluatorId"]

print(f"✅ Custom Evaluator Created!")
print(f"   Evaluator ID: {evaluator_id}")
print(f"   Name: response_quality_with_scope")
print(f"   Level: TRACE")
print(f"   Scale: Very Poor (0.0) → Very Good (1.0)")

# Store for later use
%store evaluator_id

Output()

🔧 Creating custom evaluator...



✓ Evaluator created successfully!

ID: response_quality_with_scope-915h1S4d78

ARN: arn:aws:bedrock-agentcore:us-east-1:809690530445:evaluator/response_quality_with_scope-915h1S4d78

Use: eval_client.run(evaluators=['response_quality_with_scope-915h1S4d78'])

✅ Custom Evaluator Created!
   Evaluator ID: response_quality_with_scope-915h1S4d78
   Name: response_quality_with_scope
   Level: TRACE
   Scale: Very Poor (0.0) → Very Good (1.0)
Stored 'evaluator_id' (str)


---
# Part 5: On-Demand Evaluations

Now let's evaluate the agent session we created in Part 2.

We'll run evaluations at all three levels:
- **Session level**: Goal Success Rate (whole conversation)
- **Trace level**: Correctness + our custom evaluator (per turn)
- **Span level**: Tool Selection + Parameter Accuracy (per tool call)

In [13]:
# Restore stored variables
%store -r launch_result
%store -r session_id
%store -r evaluator_id

print("📂 Loaded evaluation context:")
print(f"   Agent ID:     {launch_result.agent_id}")
print(f"   Agent ARN:    {launch_result.agent_arn}")
print(f"   Session ID:   {session_id}")
print(f"   Custom Eval:  {evaluator_id}")

from datetime import datetime, timedelta
from bedrock_agentcore_starter_toolkit.operations.observability.client import ObservabilityClient

def wait_for_session_spans(agent_id, session_id, region, attempts=12, delay_seconds=20, lookback_days=1):
    """Block until the session spans are visible in AgentCore Observability."""
    obs_client = ObservabilityClient(region_name=region)
    for attempt in range(1, attempts + 1):
        end_time = datetime.now()
        start_time = end_time - timedelta(days=lookback_days)
        spans = obs_client.query_spans_by_session(
            session_id=session_id,
            start_time_ms=int(start_time.timestamp() * 1000),
            end_time_ms=int(end_time.timestamp() * 1000),
            agent_id=agent_id,
        )
        if spans:
            print(f"   Found {len(spans)} spans for session {session_id}.")
            return spans
        if attempt == attempts:
            raise RuntimeError(
                f"No spans found for session {session_id} after {attempts} checks. "
                "Check CloudWatch / AgentCore Observability ingestion."
            )
        print(f"   Spans not ready yet (attempt {attempt}/{attempts}). Waiting {delay_seconds}s...")
        time.sleep(delay_seconds)

def run_evaluation_with_retry(eval_client, agent_id, session_id, evaluators):
    wait_for_session_spans(agent_id=agent_id, session_id=session_id, region=region)
    return eval_client.run(
        agent_id=agent_id,
        session_id=session_id,
        evaluators=evaluators,
    )


📂 Loaded evaluation context:
   Agent ID:     eval_lab_agent-rOSvq15tRI
   Agent ARN:    arn:aws:bedrock-agentcore:us-east-1:809690530445:runtime/eval_lab_agent-rOSvq15tRI
   Session ID:   30d361a8-b595-4120-941d-9f7a9d960d42
   Custom Eval:  response_quality_with_scope-915h1S4d78


In [14]:
print("🎯 SESSION-LEVEL: Goal Success Rate\n")
print("Question: Did the agent complete all the user's goals across the full conversation?")
print()

goal_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.GoalSuccessRate"]
)

print(f"Results ({len(goal_results.results)} result for the whole session):\n")
for result in goal_results.results:
    display(Markdown(f"""
**Evaluator**: `{result.evaluator_name}`

**Score**: `{result.label}` ({result.value})

**Explanation**: {result.explanation}

**Tokens Used**: {result.token_usage}
"""))
    print("─" * 60)

🎯 SESSION-LEVEL: Goal Success Rate

Question: Did the agent complete all the user's goals across the full conversation?

   Found 22 spans for session 30d361a8-b595-4120-941d-9f7a9d960d42.


Evaluating session: 30d361a8-b595-4120-941d-9f7a9d960d42

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.GoalSuccessRate

Output()

Results (1 result for the whole session):




**Evaluator**: `Builtin.GoalSuccessRate`

**Score**: `No` (0.0)

**Explanation**: Let me analyze each user goal in the conversation:

1. **First goal: "How much is 2 + 2?"**
   - Expected: The assistant should use the calculator tool to compute 2 + 2 and return the result.
   - Actual: The assistant correctly used the calculator tool with expression '2 + 2', received the result '4', and responded with "The answer is **4**."
   - Status: ✓ Achieved

2. **Second goal: "What is the weather like today?"**
   - Expected: The assistant should either use the get_weather tool (if it can determine location) or ask for the location since weather queries require a location parameter.
   - Actual: The assistant asked for clarification about which location to check, which is appropriate since no location was provided. However, the user never provided a location in response, so the weather information was never retrieved.
   - Status: ✗ Not achieved - The user's goal to get weather information was not fulfilled because they didn't receive the weather data.

3. **Third goal: "Can you tell me the capital of the United States?"**
   - Expected: This is outside the scope of available tools (calculator and get_weather).
   - Actual: The assistant correctly declined and explained it can only help with math and weather.
   - Status: ✓ Appropriately handled (though the goal itself cannot be achieved with available tools)

Overall: The user's second goal was not achieved as they never received weather information.

**Tokens Used**: {'inputTokens': 1161, 'outputTokens': 381, 'totalTokens': 1542}


────────────────────────────────────────────────────────────


In [15]:
print("✅ TRACE-LEVEL: Correctness\n")
print("Question: Is each individual agent response factually correct?")
print("(One result per user turn)")
print()

correctness_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness"]
)

print(f"Results ({len(correctness_results.results)} results — one per turn):\n")
for i, result in enumerate(correctness_results.results, 1):
    display(Markdown(f"""
**Turn {i}** | **{result.evaluator_name}**

**Score**: `{result.label}` ({result.value})

**Explanation**: {result.explanation}
"""))
    print("─" * 60)

✅ TRACE-LEVEL: Correctness

Question: Is each individual agent response factually correct?
(One result per user turn)

   Found 22 spans for session 30d361a8-b595-4120-941d-9f7a9d960d42.


Evaluating session: 30d361a8-b595-4120-941d-9f7a9d960d42

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.Correctness

Output()

...

────────────────────────────────────────────────────────────



**Turn 2** | **Builtin.Correctness**

**Score**: `Perfectly Correct` (1.0)

**Explanation**: The task is to evaluate whether the Assistant's response is correct and accurate. Looking at the context, the user asks 'What is the weather like today?' The Assistant responds by asking for clarification about which location the user wants weather information for.

This is a reasonable and appropriate response because:
1. Weather information is location-specific
2. The user did not provide a location in their query
3. The Assistant cannot provide accurate weather information without knowing the location
4. Asking for clarification is the correct approach to handle this ambiguous request

The response is polite, helpful, and logically sound. It acknowledges the request and explains what additional information is needed to fulfill it. There are no factual errors or incorrect statements in the response. The Assistant is handling the query appropriately by requesting necessary information before attempting to provide weather data.

Since the response correctly identifies the missing information (location) and appropriately asks for clarification rather than making assumptions or providing incorrect/generic information, this is a correct response to the user's query.


────────────────────────────────────────────────────────────



**Turn 3** | **Builtin.Correctness**

**Score**: `Incorrect` (0.0)

**Explanation**: The task is to evaluate whether the Assistant's response correctly answers the question 'Can you tell me the capital of the United States?'

The correct answer to this question is 'Washington, D.C.' (or variations like 'Washington D.C.' or 'Washington, District of Columbia').

The Assistant's response does not provide this answer. Instead, it states that it is designed to help with math calculations and weather information only, and that questions about geography are outside its area of focus.

While the Assistant may be constrained to certain capabilities based on its design, the evaluation criteria specified that we should focus on whether the response correctly answers the task at hand - which is a straightforward factual question about the capital of the United States.

The Assistant failed to provide the correct answer (Washington, D.C.) and instead declined to answer the question. Therefore, the response is incorrect as it does not provide the requested information.


────────────────────────────────────────────────────────────


In [16]:
print("🔧 SPAN-LEVEL: Tool Selection & Parameter Accuracy\n")
print("Question: Did the agent choose the right tools with the right parameters?")
print("(One result per tool call — multiple possible per turn)")
print()

tool_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[
        "Builtin.ToolSelectionAccuracy",
        "Builtin.ToolParameterAccuracy"
    ]
)

print(f"Results ({len(tool_results.results)} results — one per tool invocation per metric):\n")
for result in tool_results.results:
    display(Markdown(f"""
**Metric**: `{result.evaluator_name}`

**Score**: `{result.label}` ({result.value})

**Explanation**: {result.explanation}

**Context**: {str(result.context)[:200]}
"""))
    print("─" * 60)

🔧 SPAN-LEVEL: Tool Selection & Parameter Accuracy

Question: Did the agent choose the right tools with the right parameters?
(One result per tool call — multiple possible per turn)

   Found 22 spans for session 30d361a8-b595-4120-941d-9f7a9d960d42.


Evaluating session: 30d361a8-b595-4120-941d-9f7a9d960d42

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.ToolSelectionAccuracy, Builtin.ToolParameterAccuracy

Output()

Results (2 results — one per tool invocation per metric):




**Metric**: `Builtin.ToolSelectionAccuracy`

**Score**: `Yes` (1.0)

**Explanation**: The user explicitly asked 'How much is 2 + 2?' which is a straightforward arithmetic question. The assistant has access to a calculator tool that can perform mathematical calculations. The action taken was to call the calculator with the expression '2 + 2', which directly corresponds to the user's question. The tool successfully returned the result '4', which is the correct answer to the user's query. This action is clearly justified because: (1) It directly addresses the user's explicit request for a mathematical calculation, (2) The action is perfectly aligned with the user's intent to get the sum of 2 + 2, (3) All necessary parameters (the expression '2 + 2') are available and correctly formatted, and (4) A helpful assistant would reasonably use available computational tools to provide accurate answers to mathematical questions. While this is a simple calculation that could potentially be answered without a tool, using the calculator ensures accuracy and is a reasonable approach for an AI assistant to take when mathematical computation tools are available.

**Context**: {'spanContext': {'sessionId': '30d361a8-b595-4120-941d-9f7a9d960d42', 'traceId': '69df770613e95cdb48148dc75804d045', 'spanId': '77956a756dde3bd9'}}


────────────────────────────────────────────────────────────



**Metric**: `Builtin.ToolParameterAccuracy`

**Score**: `Yes` (1.0)

**Explanation**: The tool-call uses the 'calculator' tool with a single parameter 'expression' set to '2 + 2'. 

Analyzing the parameter source:
- The user explicitly asked "How much is 2 + 2?" in the conversation history
- The expression parameter value '2 + 2' is directly extracted from the user's question
- This is a faithful representation of what the user requested to calculate
- The parameter format appears correct as a string containing a mathematical expression
- No values were fabricated or hallucinated - the exact numbers and operation (2 + 2) come directly from the user's input

The parameter faithfully uses information provided by the user without adding, modifying, or fabricating any values. The expression '2 + 2' matches exactly what the user asked to calculate.

**Context**: {'spanContext': {'sessionId': '30d361a8-b595-4120-941d-9f7a9d960d42', 'traceId': '69df770613e95cdb48148dc75804d045', 'spanId': '77956a756dde3bd9'}}


────────────────────────────────────────────────────────────


In [17]:
print("🎨 TRACE-LEVEL: Custom Evaluator (5-level + scope enforcement)\n")
print("Question: Is the response high quality AND does the agent stay in scope?")
print()
print("Expected behavior:")
print("  • Math turn (2+2)    → Very Good or Good (correct, in-scope)")
print("  • Weather turn       → Good or OK (tool used correctly, in-scope)")
print("  • Capital turn       → Very Poor (out-of-scope! capital is not weather or math)")
print()

custom_results = run_evaluation_with_retry(
    eval_client,
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[evaluator_id]
)

print(f"Results ({len(custom_results.results)} results — one per turn):\n")
for i, result in enumerate(custom_results.results, 1):
    score_value = result.value if result.value is not None else 0
    score_text = result.value if result.value is not None else "n/a"
    score_emoji = "✅" if score_value >= 0.75 else ("⚠️" if score_value >= 0.5 else "❌")
    display(Markdown(f"""
{score_emoji} **Turn {i}** | `{result.evaluator_name}`

**Score**: `{result.label}` ({score_text})

**Explanation**: {result.explanation}
"""))
    print("─" * 60)

print()
print("💡 Did the third turn get 'Very Poor'?")
print("   That's our custom scope rule kicking in — the agent answered geography, not math/weather.")

🎨 TRACE-LEVEL: Custom Evaluator (5-level + scope enforcement)

Question: Is the response high quality AND does the agent stay in scope?

Expected behavior:
  • Math turn (2+2)    → Very Good or Good (correct, in-scope)
  • Weather turn       → Good or OK (tool used correctly, in-scope)
  • Capital turn       → Very Poor (out-of-scope! capital is not weather or math)

   Found 22 spans for session 30d361a8-b595-4120-941d-9f7a9d960d42.


Evaluating session: 30d361a8-b595-4120-941d-9f7a9d960d42

Mode: All traces (most recent 1000 spans)

Evaluators: response_quality_with_scope-915h1S4d78

Output()

...

────────────────────────────────────────────────────────────



✅ **Turn 2** | `response_quality_with_scope`

**Score**: `Very Good` (1.0)

**Explanation**: The user asked 'What is the weather like today?' which is clearly a weather-related question and falls within the assistant's defined scope (weather and mathematical calculations). The assistant's response appropriately acknowledges this is a weather question and offers to help. However, the response asks for a location to provide the weather information. This is a reasonable and necessary follow-up question since weather information requires a specific location. The assistant is handling the question correctly by staying within scope and requesting the required information to provide an accurate answer. The response is polite, clear, and on-topic. Since the assistant is appropriately handling a weather question (which is within its scope), this should not be penalized. The response is accurate in its approach - it cannot provide weather information without knowing the location. This is a complete and appropriate response to an incomplete question. The assistant remains within its allowed scope and handles the interaction professionally.


────────────────────────────────────────────────────────────



✅ **Turn 3** | `response_quality_with_scope`

**Score**: `Very Good` (1.0)

**Explanation**: The user asked about the capital of the United States, which is a geography question and clearly outside the assistant's defined scope of weather and mathematical calculations. According to the IMPORTANT SCOPE RULE, if the assistant answers questions outside this scope, it must receive a 'Very Poor' rating regardless of response quality. However, in this case, the assistant did NOT answer the geography question. Instead, it explicitly declined to answer, stating 'I appreciate the question, but I'm designed to help with math calculations and weather information only. Questions about geography, history, and other topics are outside my area of focus.' The assistant correctly recognized that the question was outside its scope and refused to answer it, while politely redirecting the user to topics it can help with (weather and math). This is the appropriate behavior for a scoped assistant. The assistant stayed within its boundaries and did not provide information outside its defined scope. Therefore, this response should be rated highly as it demonstrates proper scope adherence and clear communication about its limitations.


────────────────────────────────────────────────────────────

💡 Did the third turn get 'Very Poor'?
   That's our custom scope rule kicking in — the agent answered geography, not math/weather.


In [18]:
# Save evaluation results to a JSON file
print("💾 Saving evaluation results to file...\n")

os.makedirs("eval_results", exist_ok=True)

save_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        evaluator_id
    ],
    output="eval_results/lab_evaluation_output.json"
)

# Check file was saved
if os.path.exists("eval_results/lab_evaluation_output.json"):
    with open("eval_results/lab_evaluation_output.json") as f:
        saved = json.load(f)
    print(f"✅ Results saved to: eval_results/lab_evaluation_output.json")
else:
    print("ℹ️  Results may have been saved to a different location.")
    print("   Check the eval_client.run() output above for the file path.")

💾 Saving evaluation results to file...



Evaluating session: 30d361a8-b595-4120-941d-9f7a9d960d42

Mode: All traces (most recent 1000 spans)

Evaluators: Builtin.GoalSuccessRate, Builtin.Correctness, response_quality_with_scope-915h1S4d78

Output()

✓ Results saved to: eval_results/lab_evaluation_output.json

✓ Input data saved to: eval_results/lab_evaluation_output_input.json

✅ Results saved to: eval_results/lab_evaluation_output.json


---
# Part 6: Online Evaluation — Continuous Production Monitoring

Now let's configure **online evaluation** — this runs automatically against live traffic.

Online evaluation is the production monitoring mode. Once configured, it evaluates your agent continuously based on the sampling rate you define.

In [19]:
print("⚙️ Creating Online Evaluation Configuration...\n")

online_response = eval_client.create_online_config(
    agent_id=launch_result.agent_id,
    config_name="pumping_code_quality_monitor",
    sampling_rate=100,   # 100% for this lab; use 10-20% in production
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolParameterAccuracy",
        "Builtin.ToolSelectionAccuracy",
        evaluator_id  # our custom scope evaluator
    ],
    config_description=(
        "Continuous quality monitoring for weather+math assistant. "
        "Checks correctness, tool accuracy, and scope adherence."
    ),
    auto_create_execution_role=True
)

online_config_id = online_response["onlineEvaluationConfigId"]

print(f"✅ Online Evaluation Config Created!")
print(f"   Config ID: {online_config_id}")
print(f"   Sampling Rate: 100% (all sessions)")
print(f"   Evaluators: 5 (4 built-in + 1 custom)")
print(f"   Status: ENABLED — evaluating live traffic automatically")

%store online_config_id

Creating online evaluation config: pumping_code_quality_monitor for agent: eval_lab_agent-rOSvq15tRI
Configuration: sampling_rate=100.0%, evaluators=['Builtin.GoalSuccessRate', 'Builtin.Correctness', 'Builtin.ToolParameterAccuracy', 'Builtin.ToolSelectionAccuracy', 'response_quality_with_scope-915h1S4d78']
Creating online evaluation config: pumping_code_quality_monitor for agent: eval_lab_agent-rOSvq15tRI
Auto-creating execution role for config: pumping_code_quality_monitor
Getting or creating evaluation execution role for config: pumping_code_quality_monitor
Using AWS region: us-east-1, account ID: 809690530445
Role name: AgentCoreEvalsSDK-us-east-1-6f71d64bb2


⚙️ Creating Online Evaluation Configuration...



✅ Reusing existing evaluation execution role: arn:aws:iam::809690530445:role/AgentCoreEvalsSDK-us-east-1-6f71d64bb2
✓ Execution role ready: arn:aws:iam::809690530445:role/AgentCoreEvalsSDK-us-east-1-6f71d64bb2
✓ Online evaluation config created: pumping_code_quality_monitor-dOGQ2CBKqd
✓ Online evaluation config created successfully
Config ID: pumping_code_quality_monitor-dOGQ2CBKqd
Status: CREATING


✅ Online evaluation configuration created!

✅ Online Evaluation Config Created!
   Config ID: pumping_code_quality_monitor-dOGQ2CBKqd
   Sampling Rate: 100% (all sessions)
   Evaluators: 5 (4 built-in + 1 custom)
   Status: ENABLED — evaluating live traffic automatically
Stored 'online_config_id' (str)


In [20]:
# Verify the config is active
config_details = eval_client.get_online_config(config_id=online_config_id)

print("📋 Online Evaluation Config Details:")
print(json.dumps(config_details, indent=2, default=str))

📋 Online Evaluation Config Details:
{
  "ResponseMetadata": {
    "RequestId": "87e59e91-e422-4647-aa05-56ce6400e51a",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Wed, 15 Apr 2026 11:35:59 GMT",
      "content-type": "application/json",
      "content-length": "1220",
      "connection": "keep-alive",
      "x-amzn-requestid": "87e59e91-e422-4647-aa05-56ce6400e51a",
      "x-amz-apigw-id": "b2-1AHHlIAMELrw=",
      "x-amzn-trace-id": "Root=1-69df781f-26f97c78793ef70d5daf08e6"
    },
    "RetryAttempts": 0
  },
  "onlineEvaluationConfigArn": "arn:aws:bedrock-agentcore:us-east-1:809690530445:online-evaluation-config/pumping_code_quality_monitor-dOGQ2CBKqd",
  "onlineEvaluationConfigId": "pumping_code_quality_monitor-dOGQ2CBKqd",
  "onlineEvaluationConfigName": "pumping_code_quality_monitor",
  "description": "Continuous quality monitoring for weather+math assistant. Checks correctness, tool accuracy, and scope adherence.",
  "rule": {
    "samplingConfig": {
      "sam

In [21]:
# Invoke the agent multiple times to trigger online evaluation
agentcore_data_client = boto3.client("bedrock-agentcore", region_name=region)

def invoke_agent_direct(agent_arn, prompt):
    """Invoke the deployed agent via boto3."""
    response = agentcore_data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        qualifier="DEFAULT",
        payload=json.dumps({"prompt": prompt})
    )
    events = []
    for event in response.get("response", []):
        if hasattr(event, "read"):
            events.append(event.read().decode("utf-8"))
        elif isinstance(event, bytes):
            events.append(event.decode("utf-8"))
    return events[0] if events else "{}"

print("📡 Triggering new agent sessions to activate online evaluation...\n")
print("   These new sessions will be automatically evaluated by our online config.")
print()

online_prompts = [
    "How much is 7 + 9 + 10 * 2?",
    "Is it raining right now?",
    "What is 20% of 300?",
    "What can you help me with?",
    "What is the capital of New York State?"  # out-of-scope!
]

for prompt in online_prompts:
    print(f"   → {prompt}")
    try:
        result = invoke_agent_direct(launch_result.agent_arn, prompt)
        print(f"     ✅ Response received")
    except Exception as e:
        print(f"     ⚠️  {str(e)[:60]}")

print()
print("✅ Agent invocations complete!")
print("   Online evaluation is now processing these sessions in the background.")
print("   Results appear in CloudWatch under AgentCore Observability.")
print()
print("📊 Where to view results:")
print("   AWS Console → CloudWatch → Gen AI Observability → AgentCore → Agents")
print("   → Select your agent → DEFAULT endpoint → Evaluation tab")
print("   (May take 2-5 minutes to appear)")

📡 Triggering new agent sessions to activate online evaluation...

   These new sessions will be automatically evaluated by our online config.

   → How much is 7 + 9 + 10 * 2?
     ✅ Response received
   → Is it raining right now?
     ✅ Response received
   → What is 20% of 300?
     ✅ Response received
   → What can you help me with?
     ✅ Response received
   → What is the capital of New York State?
     ✅ Response received

✅ Agent invocations complete!
   Online evaluation is now processing these sessions in the background.
   Results appear in CloudWatch under AgentCore Observability.

📊 Where to view results:
   AWS Console → CloudWatch → Gen AI Observability → AgentCore → Agents
   → Select your agent → DEFAULT endpoint → Evaluation tab
   (May take 2-5 minutes to appear)


---
# Part 7: Interpreting Your Results — Analysis

Let's build a simple analysis of the evaluation results we collected.

In [22]:
print("📊 EVALUATION SUMMARY ANALYSIS\n")
print("=" * 60)

# Aggregate all results
all_evaluations = {
    "Builtin.GoalSuccessRate": goal_results,
    "Builtin.Correctness": correctness_results,
    "Tool Metrics": tool_results,
    "Custom Scope Evaluator": custom_results
}

for eval_name, results_obj in all_evaluations.items():
    results = results_obj.results
    if not results:
        continue
    
    values = [r.value for r in results if r.value is not None]
    avg = sum(values) / len(values) if values else 0
    
    print(f"\n  📌 {eval_name}")
    print(f"     Results: {len(results)}")
    print(f"     Average Score: {avg:.2f}")
    print(f"     Labels:")
    for r in results:
        score_value = r.value if r.value is not None else 0
        score_text = f"{r.value:.2f}" if r.value is not None else "n/a"
        bar = "█" * int(score_value * 10) + "░" * (10 - int(score_value * 10))
        evaluator_short = r.evaluator_name.split(".")[-1] if r.evaluator_name else "?"
        print(f"       [{bar}] {r.label} ({score_text}) — {evaluator_short}")

print()
print("=" * 60)
print("\n💡 What to look for:")
print("   • Goal Success Rate < 0.7 → Agent isn't completing user tasks")
print("   • Correctness < 0.8 → Factual accuracy issues in responses")
print("   • Tool accuracy < 0.8 → Agent is misusing tools (check params)")
print("   • Custom score for capital question → should be Very Poor (0.0)")
print("   • This out-of-scope detection would be missed by Builtin.Correctness")

📊 EVALUATION SUMMARY ANALYSIS


  📌 Builtin.GoalSuccessRate
     Results: 1
     Average Score: 0.00
     Labels:
       [░░░░░░░░░░] No (0.00) — GoalSuccessRate

  📌 Builtin.Correctness
     Results: 3
     Average Score: 0.67
     Labels:
       [██████████] Perfectly Correct (1.00) — Correctness
       [██████████] Perfectly Correct (1.00) — Correctness
       [░░░░░░░░░░] Incorrect (0.00) — Correctness

  📌 Tool Metrics
     Results: 2
     Average Score: 1.00
     Labels:
       [██████████] Yes (1.00) — ToolSelectionAccuracy
       [██████████] Yes (1.00) — ToolParameterAccuracy

  📌 Custom Scope Evaluator
     Results: 3
     Average Score: 1.00
     Labels:
       [██████████] Very Good (1.00) — response_quality_with_scope
       [██████████] Very Good (1.00) — response_quality_with_scope
       [██████████] Very Good (1.00) — response_quality_with_scope


💡 What to look for:
   • Goal Success Rate < 0.7 → Agent isn't completing user tasks
   • Correctness < 0.8 → Factual accur

---
# Part 8: 🧹 Cleanup

Clean up all resources to avoid ongoing costs.

**Note**: The deployed AgentCore Runtime agent will also stop incurring costs when idle (after the `idle_timeout` period).

In [23]:
print("🧹 Starting cleanup...\n")

# ── Step 1: Delete online evaluation config ───────────────────────────────
try:
    print("Step 1: Deleting online evaluation config...")
    %store -r online_config_id
    eval_client.delete_online_config(config_id=online_config_id)
    print(f"   ✅ Online config deleted: {online_config_id}")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 2: Delete custom evaluator ──────────────────────────────────────
try:
    print("\nStep 2: Deleting custom evaluator...")
    %store -r evaluator_id
    eval_client.delete_evaluator(evaluator_id=evaluator_id)
    print(f"   ✅ Custom evaluator deleted: {evaluator_id}")
except Exception as e:
    print(f"   ⚠️  {str(e)[:80]}")

# ── Step 3: Delete the deployed agent ────────────────────────────────────
try:
    print("\nStep 3: Deleting AgentCore Runtime agent...")
    %store -r launch_result
    agent_id = getattr(launch_result, "agent_id", None)
    ctrl = boto3.client("bedrock-agentcore-control", region_name=region)

    if agent_id:
        ctrl.delete_agent_runtime(agentRuntimeId=agent_id)
        print(f"   Delete requested for agent runtime: {agent_id}")
    else:
        raise RuntimeError("launch_result.agent_id is missing")

    for attempt in range(1, 21):
        matching = [
            item for item in ctrl.list_agent_runtimes(maxResults=100).get("agentRuntimes", [])
            if item.get("agentRuntimeId") == agent_id
        ]
        if not matching:
            print(f"   ✅ Agent {agent_id} deleted")
            break
        status = matching[0].get("status")
        print(f"   Waiting for deletion... current status: {status}")
        time.sleep(10)
    else:
        print(f"   ⚠️  Agent {agent_id} still exists after waiting. Status: {matching[0].get('status')}")
except Exception as e:
    print(f"   ⚠️  {str(e)[:120]}")
    print("   You can also delete from the AgentCore console manually.")

# ── Step 4: Clean up local files ─────────────────────────────────────────
import shutil
for fname in [AGENT_FILE, REQUIREMENTS_FILE, RUNTIME_WORK_DIR / ".bedrock_agentcore.yaml"]:
    if os.path.exists(fname):
        os.remove(fname)

print("\n🎉 Cleanup complete!")
print("   eval_results/ folder retained — check lab_evaluation_output.json for your results.")

Deleting online evaluation config: pumping_code_quality_monitor-dOGQ2CBKqd


🧹 Starting cleanup...

Step 1: Deleting online evaluation config...


✓ Online evaluation config deleted successfully


✅ Configuration deleted!

Output()

   ✅ Online config deleted: pumping_code_quality_monitor-dOGQ2CBKqd

Step 2: Deleting custom evaluator...


✓ Evaluator deleted successfully

   ✅ Custom evaluator deleted: response_quality_with_scope-915h1S4d78

Step 3: Deleting AgentCore Runtime agent...
   Delete requested for agent runtime: eval_lab_agent-rOSvq15tRI
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   Waiting for deletion... current status: DELETING
   ✅ Agent eval_lab_agent-rOSvq15tRI deleted

🎉 Cleanup complete!
   eval_results/ folder retained — check lab_evaluation_output.json for your results.


---
# 🎉 Lab Complete!

## What You Accomplished

| Step | Concept | What You Learned |
|------|---------|------------------|
| Part 2 | Deploy agent to Runtime | OTEL instrumentation is automatic |
| Part 3 | Explore built-in evaluators | 13 pre-built metrics, fixed configs |
| Part 4 | Custom evaluator (5-level) | LLM-as-judge with scope enforcement |
| Part 5 | On-demand: GoalSuccessRate | Session-level task completion |
| Part 5 | On-demand: Correctness | Trace-level per-turn accuracy |
| Part 5 | On-demand: Tool metrics | Span-level tool usage quality |
| Part 5 | On-demand: Custom | Scope penalization in action |
| Part 6 | Online evaluation | Continuous automatic monitoring |
| Part 7 | Results analysis | How to read and act on scores |

## Key Takeaways

- 📊 **Three levels**: Span (tool calls) → Trace (turns) → Session (full conversation)
- 🤖 **13 built-in evaluators** cover most use cases immediately
- 🎨 **Custom evaluators** enable domain-specific and scope-aware quality checks
- 💡 **The explanation field** is the most valuable field — read it to understand WHY
- 🔄 **On-demand + Online** complement each other: dev-time testing + prod monitoring
